# Stretch and transpose peak tables

`conga_multidft` is an events table; `bayle1` is a snapshot. This notebook
plays the snapshot, freezes it in time, splits it, then stretches and transposes
each segment, and listens to one result. Read `visual_tsv` first for the table
kinds.

In [ ]:
from pathlib import Path
import audiospylt as asp

## Kinds

`infer_kind` returns `"events"` or `"snapshot"` from the column set.

In [ ]:
events = asp.read_table("../data/tables/conga_multidft.tsv")
snapshot = asp.read_table("../data/tables/bayle1.tsv")
print("conga_multidft:", asp.infer_kind(events), f"({len(events)} rows)")
print("bayle1:", asp.infer_kind(snapshot), f"({len(snapshot)} rows)")

## Snapshot plot

`bayle1` holds 64 partials from 150.0 Hz to 9781.6 Hz.

In [ ]:
asp.nb.show(asp.viz.plot_snapshot(snapshot, labels=["bayle1"]))
print(f"{len(snapshot)} peaks, {snapshot.freq.min():.1f}–{snapshot.freq.max():.1f} Hz")

## Listen to the frame

This is `bayle1` before any processing, held for 2.0 s with `snapshot_to_events`
(next section): no attack and no decay. It plays at its own level, as the result
at the end of this notebook does, so the two compare directly.

In [ ]:
frame_audio = asp.synthesize(asp.snapshot_to_events(snapshot, 0.0, 2.0), sr=44100)
asp.nb.play(frame_audio, 44100)

## Freeze to events

A snapshot has no duration. `snapshot_to_events(snapshot, 0.0, 2.0)` holds every
partial steady for 2.0 s. Both tables are shown below.

In [ ]:
frozen = asp.snapshot_to_events(snapshot, 0.0, 2.0)
print("snapshot columns:", list(snapshot.columns))
print("events columns:", list(frozen.columns))
print(f"frozen rows={len(frozen)}, duration={frozen.time_stop.iloc[0]:.1f} s")
snapshot.head()

In [ ]:
frozen.head()

## 3-D view of the freeze

All 64 partials run as horizontal lines from 0 s to 2 s.

In [ ]:
asp.nb.show(asp.viz.plot_events_3d(frozen, labels=["bayle 0–2 s"]))

## Split at cue times

`split_events(events, [0.5, 1.5])` cuts every row at those times, so the cues
become 0.0, 0.5 and 1.5 s (three segments covering 0–2 s).

In [ ]:
split = asp.split_events(frozen, [0.5, 1.5])
cues = sorted(split.time_start.unique())
print("cues (s):", [float(c) for c in cues])
print("rows per cue:", split.groupby("time_start").size().to_dict())
asp.nb.show(asp.viz.plot_events_3d(split, labels=["split"]))

## Per-segment stretch and transpose

`stretch` and `transpose` accept one value per ascending distinct `time_start`.
Factors `[1.0, 2.0, 0.5]` leave the first segment, double the middle, and halve
the last. Cents `[0, 1500, -250]` then shift those three segments.

Before you run the cell: how long is the result after those stretch factors?

In [ ]:
stretched = asp.stretch(split, [1.0, 2.0, 0.5])
processed = asp.transpose(stretched, cents=[0, 1500, -250])
spans = processed.groupby("time_start")["time_stop"].max().reset_index()
print(spans.to_string(index=False))
print(f"1500 cents -> factor {2 ** (1500 / 1200):.4f}; duration {processed.time_stop.max():.2f} s")
asp.nb.show(asp.viz.plot_events_3d(processed, labels=["stretch+transpose"]))

The timeline becomes 0–0.5 s, 0.5–2.5 s, 2.5–2.75 s (total 2.75 s). The middle
segment is up 1500 cents (frequency factor 2.3784); the last is down 250 cents.

## Partials past Nyquist

`transpose(snapshot, 10, …)` multiplies every frequency by 10. With
`fmax = sr / 2` at `sr = 44100`, before you run the cell: what happens to
partials pushed above Nyquist under `'clip'`, `'drop'` and `'keep'`?

In [ ]:
sr = 44100
nyquist = sr / 2
print(f"Nyquist {nyquist} Hz")
for mode in ("clip", "drop", "keep"):
    out = asp.transpose(snapshot, 10, fmax=nyquist, out_of_range=mode)
    above = int((out.freq > nyquist).sum()) if len(out) else 0
    print(f"{mode}: {len(out)} rows, max {out.freq.max():.1f} Hz, above Nyquist={above}")

Each call warns that 34 rows fell outside the pitch bounds. `'clip'` keeps all
64 rows and clamps the 34 overshoots to 22050.0 Hz. `'drop'` removes those 34
rows and leaves 30. `'keep'` retains all 64 scaled values, so 34 sit above
Nyquist (max 97815.8 Hz).

## Listen and write

Synthesize the stretched and transposed events at 44100 Hz. Listen for the
longer middle segment and the pitch jumps at the cue times. The middle segment's
top partials are 9781.6 Hz × 2.3784 = 23264.6 Hz, above Nyquist: `synthesize`
warns that 2 events are silent. Under `'clip'` a partial sits exactly at
`sr / 2`, where `synthesize` is silent too, so `'drop'` and `'clip'` sound the
same before you change `nyquist` (`'keep'` matters with `nyquist="alias"` in
`resynth`).

In [ ]:
audio = asp.synthesize(processed, sr=44100)
nyquist, factor = 22050.0, 2 ** (1500 / 1200)
top = float(snapshot.freq.max())
n_hi = int(((processed.freq_start >= nyquist) | (processed.freq_stop >= nyquist)).sum())
print(f"duration {len(audio) / 44100:.2f} s; events ≥ Nyquist: {n_hi}")
print(f"top mid partial {top:.1f}×{factor:.4f}={top * factor:.1f} Hz")
asp.nb.show(asp.viz.plot_waveform(audio, 44100))
asp.nb.play(audio, 44100)
Path("tsv").mkdir(exist_ok=True)
path = asp.write_table(processed, "tsv/bayle_stretch_transpose.tsv")
print(path)
print(path.read_text().splitlines()[0])